# Working ArXiv Search with Transformers

This notebook combines arXiv data processing with transformer-based semantic search.

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import requests
from datetime import datetime
from io import StringIO
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import json
import d6tflow
print("Libraries imported successfully!")

Welcome to d6tflow! For Q&A see https://github.com/d6t/d6tflow
Libraries imported successfully!


In [2]:
# Configuration
query_words = ['machine', 'learning', 'neural', 'network', 'transformer', 'attention']
query_size = 1000  # Start smaller for testing

# Build query URL
queries = [word + '&' for word in query_words]
query = ''.join(queries)
url = f'https://export.arxiv.org/api/query?search_query=all:{query}start=0&max_results={query_size}'
print(f"Query URL: {url}")

search_phrase = ' '.join(query_words[:2])  # "machine learning"
print(f"Search phrase: {search_phrase}")

Query URL: https://export.arxiv.org/api/query?search_query=all:machine&learning&neural&network&transformer&attention&start=0&max_results=1000
Search phrase: machine learning


In [3]:
# Fetch data from arXiv
print("Fetching data from arXiv...")
response = requests.get(url)
xml_data = response.text
print(f"Fetched {len(xml_data)} characters of XML data")

Fetching data from arXiv...


Fetched 1961842 characters of XML data


In [4]:
# Parse and process the data
print("Processing arXiv data...")

# Parse XML
df = pd.read_xml(StringIO(xml_data))
print(f"Parsed XML into DataFrame with {len(df)} rows")

# Process data (skip first 7 entries which are usually metadata)
if len(df) > 7:
    papers_df = pd.DataFrame()
    papers_df['title'] = df['title'][7:].reset_index(drop=True)
    papers_df['abstract'] = df['summary'][7:].reset_index(drop=True)
    papers_df['published'] = pd.to_datetime(df['published'][7:].reset_index(drop=True))
    papers_df['updated'] = pd.to_datetime(df['updated'][7:].reset_index(drop=True))
    papers_df['url'] = df['id'][7:].reset_index(drop=True)
    
    # Add analysis columns
    two_years_ago = pd.Timestamp.now(tz='UTC') - pd.DateOffset(years=2)
    papers_df['is_recent'] = papers_df['published'].apply(lambda x: x > two_years_ago)
    papers_df['title_has_keywords'] = papers_df['title'].str.contains(search_phrase, case=False, na=False)
    papers_df['combined_text'] = papers_df['title'] + ' ' + papers_df['abstract']
    
    # Clean up
    papers_df = papers_df.dropna(subset=['title', 'abstract'])
    
    print(f"Processed {len(papers_df)} papers")
    print(f"Recent papers (last 2 years): {papers_df['is_recent'].sum()}")
    print(f"Papers with keywords in title: {papers_df['title_has_keywords'].sum()}")
else:
    print("Warning: Not enough data entries")
    papers_df = pd.DataFrame()

Processing arXiv data...
Parsed XML into DataFrame with 1007 rows
Processed 1000 papers
Recent papers (last 2 years): 128
Papers with keywords in title: 433


In [5]:
# Generate embeddings using transformer model
if len(papers_df) > 0:
    print("Loading transformer model...")
    model = SentenceTransformer('paraphrase-albert-small-v2')
    
    print(f"Generating embeddings for {len(papers_df)} papers...")
    texts = papers_df['combined_text'].tolist()
    embeddings = model.encode(texts, show_progress_bar=True)
    
    print(f"Generated embeddings with shape: {embeddings.shape}")
else:
    print("No papers to process")
    model = None
    embeddings = np.array([])

Loading transformer model...


Generating embeddings for 1000 papers...


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Generated embeddings with shape: (1000, 768)


In [6]:
# Define semantic search function
def semantic_search(query, top_k=5):
    """Search for papers similar to the query"""
    if len(papers_df) == 0 or model is None:
        print("No papers available for search")
        return []
    
    # Generate embedding for query
    query_embedding = model.encode([query])
    
    # Calculate similarities
    similarities = cosine_similarity(query_embedding, embeddings)[0]
    
    # Get top k results
    top_indices = np.argsort(similarities)[::-1][:top_k]
    
    results = []
    for idx in top_indices:
        result = {
            'title': papers_df.iloc[idx]['title'],
            'abstract': papers_df.iloc[idx]['abstract'],
            'published': papers_df.iloc[idx]['published'],
            'url': papers_df.iloc[idx]['url'],
            'similarity_score': similarities[idx]
        }
        results.append(result)
    
    return results

def display_results(results, max_abstract_length=200):
    """Display search results"""
    for i, paper in enumerate(results, 1):
        print(f"\n{i}. {paper['title']}")
        print(f"   Similarity: {paper['similarity_score']:.3f}")
        print(f"   Published: {str(paper['published'])[:10]}")
        
        abstract = paper['abstract']
        if len(abstract) > max_abstract_length:
            abstract = abstract[:max_abstract_length] + "..."
        print(f"   Abstract: {abstract}")
        print(f"   URL: {paper['url']}")
        print("-" * 80)

print("Search functions defined!")

Search functions defined!


In [7]:
# Example searches - Modern AI Topics
search_queries = [
    "machine learning",
    "computer vision and image recognition",
    "reinforcement learning algorithms"
]

for query in search_queries:
    print(f"\n{'='*60}")
    print(f"SEARCH QUERY: {query}")
    print(f"{'='*60}")
    
    results = semantic_search(query, top_k=3)
    display_results(results)


SEARCH QUERY: machine learning

1. Machine Learning in Network Security Using KNIME Analytics
   Similarity: 0.663
   Published: 2019-11-18
   Abstract:   Machine learning has more and more effect on our every day's life. This field
keeps growing and expanding into new areas. Machine learning is based on the
implementation of artificial intelligence t...
   URL: http://arxiv.org/abs/2001.11489v1
--------------------------------------------------------------------------------

2. Machine learning and deep learning
   Similarity: 0.643
   Published: 2021-04-12
   Abstract:   Today, intelligent systems that offer artificial intelligence capabilities
often rely on machine learning. Machine learning describes the capacity of
systems to learn from problem-specific training ...
   URL: http://arxiv.org/abs/2104.05314v2
--------------------------------------------------------------------------------

3. Machine Learning and Computational Mathematics
   Similarity: 0.642
   Published: 2020-09-

## 🤖 Large Language Model (LLM) Focused Searches

Let's explore papers specifically related to Large Language Models and modern NLP:

In [8]:
# LLM and Language Model focused searches
llm_queries = [
    "large language models and generative AI",
    "BERT transformer architecture and pre-training",
    "GPT models and autoregressive generation"
]

print("🤖 LARGE LANGUAGE MODEL RESEARCH")
print("=" * 50)

for query in llm_queries:
    print(f"\n🔍 SEARCHING: '{query}'")
    print("-" * 50)
    
    results = semantic_search(query, top_k=2)  # Show top 2 for each
    display_results(results)

🤖 LARGE LANGUAGE MODEL RESEARCH

🔍 SEARCHING: 'large language models and generative AI'
--------------------------------------------------

1. Exploring the Use of Attention within an Neural Machine Translation
  Decoder States to Translate Idioms
   Similarity: 0.533
   Published: 2018-10-10
   Abstract:   Idioms pose problems to almost all Machine Translation systems. This type of
language is very frequent in day-to-day language use and cannot be simply
ignored. The recent interest in memory augmente...
   URL: http://arxiv.org/abs/1810.06695v1
--------------------------------------------------------------------------------

2. On the Limitations and Prospects of Machine Unlearning for Generative AI
   Similarity: 0.524
   Published: 2024-08-01
   Abstract:   Generative AI (GenAI), which aims to synthesize realistic and diverse data
samples from latent variables or other data modalities, has achieved remarkable
results in various domains, such as natural...
   URL: http://arxiv.org/a

## 📝 Natural Language Processing Focused Searches

Now let's explore traditional and modern NLP techniques:

In [9]:
# Natural Language Processing focused searches
nlp_queries = [
    "natural language understanding and sentiment analysis",
    "text summarization and information extraction",
    "machine translation and multilingual models",
    "question answering systems and reading comprehension"
]

print("📝 NATURAL LANGUAGE PROCESSING RESEARCH")
print("=" * 50)

for query in nlp_queries:
    print(f"\n🔍 SEARCHING: '{query}'")
    print("-" * 50)
    
    results = semantic_search(query, top_k=2)  # Show top 2 for each
    display_results(results)

📝 NATURAL LANGUAGE PROCESSING RESEARCH

🔍 SEARCHING: 'natural language understanding and sentiment analysis'
--------------------------------------------------



1. Thumbs up? Sentiment Classification using Machine Learning Techniques
   Similarity: 0.606
   Published: 2002-05-28
   Abstract:   We consider the problem of classifying documents not by topic, but by overall
sentiment, e.g., determining whether a review is positive or negative. Using
movie reviews as data, we find that standar...
   URL: http://arxiv.org/abs/cs/0205070v1
--------------------------------------------------------------------------------

2. RKadiyala at SemEval-2024 Task 8: Black-Box Word-Level Text Boundary
  Detection in Partially Machine Generated Texts
   Similarity: 0.491
   Published: 2024-10-22
   Abstract:   With increasing usage of generative models for text generation and widespread
use of machine generated texts in various domains, being able to distinguish
between human written and machine generated...
   URL: http://arxiv.org/abs/2410.16659v1
--------------------------------------------------------------------------------

🔍 SEARCHING: 'text summarizatio


1. RKadiyala at SemEval-2024 Task 8: Black-Box Word-Level Text Boundary
  Detection in Partially Machine Generated Texts
   Similarity: 0.391
   Published: 2024-10-22
   Abstract:   With increasing usage of generative models for text generation and widespread
use of machine generated texts in various domains, being able to distinguish
between human written and machine generated...
   URL: http://arxiv.org/abs/2410.16659v1
--------------------------------------------------------------------------------

2. Neural Machine Translation System of Indic Languages -- An Attention
  based Approach
   Similarity: 0.387
   Published: 2020-02-02
   Abstract:   Neural machine translation (NMT) is a recent and effective technique which
led to remarkable improvements in comparison of conventional machine
translation techniques. Proposed neural machine transl...
   URL: http://arxiv.org/abs/2002.02758v1
--------------------------------------------------------------------------------

🔍 SEARCHING: 'm

## 🎯 Comparing Search Results Across Domains

Let's compare how the same query performs across different AI domains:

In [10]:
# Comparison searches - same concept across different domains
comparison_queries = [
    "consciousness",
    "transfer learning",
    "fine-tuning models",
    "quantum machine"
]

print("🎯 CROSS-DOMAIN CONCEPT COMPARISON")
print("=" * 50)

for query in comparison_queries:
    print(f"\n🔍 CONCEPT: '{query}'")
    print("-" * 40)
    
    results = semantic_search(query, top_k=3)
    
    # Display just titles and similarity scores for comparison
    for i, paper in enumerate(results, 1):
        print(f"  {i}. {paper['similarity_score']:.3f} - {paper['title']}")
    print()

🎯 CROSS-DOMAIN CONCEPT COMPARISON

🔍 CONCEPT: 'transfer learning'
----------------------------------------


  1. 0.632 - Spatial Transfer Learning with Simple MLP
  2. 0.470 - Inferring Reward Machines and Transition Machines from Partially
  Observable Markov Decision Processes
  3. 0.461 - Distribution Matching for Machine Teaching


🔍 CONCEPT: 'fine-tuning models'
----------------------------------------
  1. 0.491 - On the Conditions for Domain Stability for Machine Learning: a
  Mathematical Approach
  2. 0.481 - Differential Replication in Machine Learning
  3. 0.441 - Effective dimension of machine learning models



In [11]:
# Dataset analytics
if len(papers_df) > 0:
    print("📊 Dataset Analytics")
    print("=" * 40)
    print(f"Total papers: {len(papers_df)}")
    print(f"Recent papers (last 2 years): {papers_df['is_recent'].sum()}")
    print(f"Papers with keywords in title: {papers_df['title_has_keywords'].sum()}")
    
    # Publication year distribution
    papers_df['pub_year'] = papers_df['published'].dt.year
    year_counts = papers_df['pub_year'].value_counts().sort_index()
    
    print("\n📅 Publication Year Distribution (recent years):")
    for year in sorted(year_counts.index)[-5:]:  # Last 5 years in data
        count = year_counts[year]
        print(f"  {year}: {count} papers")
    
    # Most common words in titles
    all_titles = ' '.join(papers_df['title'].str.lower())
    words = all_titles.split()
    stop_words = {'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for', 'of', 'with', 'by'}
    filtered_words = [word for word in words if len(word) > 3 and word not in stop_words]
    
    from collections import Counter
    word_counts = Counter(filtered_words)
    
    print("\n🏷️ Most Common Words in Titles:")
    for word, count in word_counts.most_common(10):
        print(f"  {word}: {count}")
else:
    print("No papers available for analytics.")

📊 Dataset Analytics
Total papers: 1000
Recent papers (last 2 years): 128
Papers with keywords in title: 433

📅 Publication Year Distribution (recent years):
  2021: 114 papers
  2022: 76 papers
  2023: 63 papers
  2024: 75 papers
  2025: 31 papers

🏷️ Most Common Words in Titles:
  machine: 656
  learning: 472
  machines: 203
  quantum: 64
  using: 59
  turing: 54
  models: 42
  learning:: 38
  survey: 35
  support: 34


## 🔬 Singular vs Plural: 'machine' vs 'machines'

Let's compare how semantic search handles singular vs plural forms. From our analytics above, we see:
- 'machine': 656 occurrences in titles
- 'machines': 203 occurrences in titles

How do these different forms affect semantic similarity in transformer embeddings?

In [12]:
# Compare singular vs plural search terms
singular_plural_queries = ['machine', 'machines']

for query in singular_plural_queries:
    print(f"\n🔍 SEMANTIC SEARCH: '{query}'")
    print("=" * 60)
    
    results = semantic_search(query, top_k=5)
    
    for i, paper in enumerate(results, 1):
        print(f"{i}. {paper['similarity_score']:.3f} - {paper['title']}")
    print()


🔍 SEMANTIC SEARCH: 'machine'


1. 0.569 - System on Programable Chip for Performance Estimation of Loom Machine
2. 0.562 - Gravitational Machines
3. 0.547 - Newtonian Mechanics Based Transient Stability PART VI: Machine
  Transformation
4. 0.544 - Expert-Augmented Machine Learning
5. 0.539 - Machines of finite depth: towards a formalization of neural networks


🔍 SEMANTIC SEARCH: 'machines'


1. 0.633 - A Tale of Two Turing Machines
2. 0.629 - Machines of finite depth: towards a formalization of neural networks
3. 0.629 - Machines as Programs: P $\neq$ NP
4. 0.612 - Recombinations of Busy Beaver Machines
5. 0.608 - The tree machine



## 🔄 Emerging AI Topic: Machine Unlearning

Let's explore the cutting-edge field of machine unlearning - the ability to make models 'forget' specific data for privacy compliance (GDPR 'right to be forgotten'):

In [13]:
# Search for machine unlearning - a hot topic in AI privacy
unlearning_query = 'machine unlearning'

print(f'🔄 EMERGING AI RESEARCH: "{unlearning_query}"')
print('=' * 70)

# First check how many actual unlearning papers we have
unlearn_papers = papers_df[papers_df['title'].str.contains('unlearn', case=False, na=False)]
privacy_papers = papers_df[papers_df['title'].str.contains('privacy', case=False, na=False)]

print(f'📊 Dataset contains {len(unlearn_papers)} papers with "unlearn" in title')
print(f'📊 Dataset contains {len(privacy_papers)} papers with "privacy" in title')

if len(unlearn_papers) > 0:
    print(f'\n🎯 ACTUAL MACHINE UNLEARNING PAPERS:')
    for i, (_, paper) in enumerate(unlearn_papers.iterrows(), 1):
        year = str(paper['published'])[:4]
        print(f'   {i}. {paper["title"]} ({year})')

# Now run semantic search
print(f'\n🔍 SEMANTIC SEARCH RESULTS:')
print('-' * 50)

results = semantic_search(unlearning_query, top_k=7)

for i, paper in enumerate(results, 1):
    print(f'{i}. {paper["similarity_score"]:.3f} - {paper["title"]}')
    print(f'   Published: {str(paper["published"])[:10]}')
    if i <= 3:  # Show abstracts for top 3
        abstract = paper['abstract'][:150] + '...' if len(paper['abstract']) > 150 else paper['abstract']
        print(f'   Abstract: {abstract}')
    print()

print('💡 MACHINE UNLEARNING CONTEXT:')
print('   • Privacy-preserving AI: Selectively removing data traces')
print('   • GDPR "right to be forgotten" compliance')
print('   • Preventing model memorization of sensitive data')
print('   • Critical for LLMs and personal data protection')

🔄 EMERGING AI RESEARCH: "machine unlearning"
📊 Dataset contains 6 papers with "unlearn" in title
📊 Dataset contains 10 papers with "privacy" in title

🎯 ACTUAL MACHINE UNLEARNING PAPERS:
   1. A Review on Machine Unlearning (2024)
   2. Verification of Machine Unlearning is Fragile (2024)
   3. Bridge the Gaps between Machine Unlearning and AI Regulation (2025)
   4. On the Limitations and Prospects of Machine Unlearning for Generative AI (2024)
   5. Mo' Memory, Mo' Problems: Stream-Native Machine Unlearning (2025)
   6. Towards Machine Unlearning Benchmarks: Forgetting the Personal
  Identities in Facial Recognition Systems (2023)

🔍 SEMANTIC SEARCH RESULTS:
--------------------------------------------------


1. 0.702 - Towards Machine Unlearning Benchmarks: Forgetting the Personal
  Identities in Facial Recognition Systems
   Published: 2023-11-03
   Abstract:   Machine unlearning is a crucial tool for enabling a classification model to
forget specific data that are used in the training time. Recently, vario...

2. 0.669 - On the Limitations and Prospects of Machine Unlearning for Generative AI
   Published: 2024-08-01
   Abstract:   Generative AI (GenAI), which aims to synthesize realistic and diverse data
samples from latent variables or other data modalities, has achieved rema...

3. 0.640 - Mo' Memory, Mo' Problems: Stream-Native Machine Unlearning
   Published: 2025-08-13
   Abstract:   Machine unlearning work assumes a static, i.i.d training environment that
doesn't truly exist. Modern ML pipelines need to learn, unlearn, and predi...

4. 0.621 - Large-scale randomized experiment reveals machine learning helps people
  learn and remember more effectively
   Published: 2020-10-09

5. 

In [14]:
# Export results
if len(papers_df) > 0:
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    
    # Save papers as CSV
    csv_filename = f"arxiv_papers_{timestamp}.csv"
    papers_df.to_csv(csv_filename, index=False)
    print(f"✅ Saved {len(papers_df)} papers to {csv_filename}")
    
    # Save recent papers as JSON
    recent_papers = papers_df[papers_df['is_recent']]
    if len(recent_papers) > 0:
        json_filename = f"recent_papers_{timestamp}.json"
        recent_papers.to_json(json_filename, orient='records', indent=2, date_format='iso')
        print(f"✅ Saved {len(recent_papers)} recent papers to {json_filename}")
else:
    print("No papers to export.")

✅ Saved 1000 papers to arxiv_papers_20250817_074228.csv
✅ Saved 128 recent papers to recent_papers_20250817_074228.json


## Interactive Search

You can now use the `semantic_search()` function to search for papers. For example:

```python
results = semantic_search("your search query here", top_k=5)
display_results(results)
```

In [15]:
# Custom search - modify this cell to search for what you want
custom_query = "attention mechanisms in transformer models"
print(f"Searching for: {custom_query}")
custom_results = semantic_search(custom_query, top_k=5)
display_results(custom_results)

Searching for: attention mechanisms in transformer models



1. How Much Can We See? A Note on Quantifying Explainability of Machine
  Learning Models
   Similarity: 0.456
   Published: 2019-10-29
   Abstract:   One of the most popular approaches to understanding feature effects of modern
black box machine learning models are partial dependence plots (PDP). These
plots are easy to understand but only able t...
   URL: http://arxiv.org/abs/1910.13376v2
--------------------------------------------------------------------------------

2. Rate-Distortion Theory in Coding for Machines and its Application
   Similarity: 0.452
   Published: 2023-05-26
   Abstract:   Recent years have seen a tremendous growth in both the capability and
popularity of automatic machine analysis of images and video. As a result, a
growing need for efficient compression methods opti...
   URL: http://arxiv.org/abs/2305.17295v2
--------------------------------------------------------------------------------

3. Unmasking Clever Hans Predictors and Assessing What Machines Re